In [48]:
!pip install -q wrds rapidfuzz pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [64]:
import wrds
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

db = wrds.Connection()  # caches credentials in ~/.pgpass after first login

Loading library list...
Done


In [65]:
# ── Pull the 18 raw Compustat line items matching the sowide dataset paper ──
fin = db.raw_sql("""
    SELECT gvkey, cik, tic, conm, sich, fyear, datadate,
           act, at, cogs, dltt, dp, ebit, oibdp, gp, invt,
           lct, ni, re, rect, revt, lt, sale, xopr,
           csho, prcc_f
    FROM comp.funda
    WHERE indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
      AND exchg IN (11, 14)
      AND fyear BETWEEN 1999 AND 2018
      AND at > 0
""", date_cols=["datadate"])

fin["mktval"] = fin["csho"] * fin["prcc_f"]
fin["gp"]    = fin["gp"].fillna(fin["sale"] - fin["cogs"])
fin["xopr"]  = fin["xopr"].fillna(fin["sale"] - fin["ebit"])
fin = fin.drop(columns=["csho", "prcc_f"])

# ── Cast fyear to int NOW so every downstream merge key is consistent ──────
fin["fyear"] = fin["fyear"].astype(int)

fin = fin.sort_values(["gvkey", "fyear"]).reset_index(drop=True)
print(fin.shape, "|", fin["gvkey"].nunique(), "companies | fyear dtype:", fin["fyear"].dtype)

(96955, 25) | 10181 companies | fyear dtype: int64


In [66]:
# ── Bankruptcy labels ─────────────────────────────────────────────────────
# dlrsn='02' = Chapter 7/11 filing; label the fiscal year before the filing date.
bk = db.raw_sql("""
    SELECT gvkey,
           EXTRACT(YEAR FROM dldte)::int - 1 AS failed_fyear
    FROM comp.company
    WHERE dlrsn = '02'
      AND dldte IS NOT NULL
      AND EXTRACT(YEAR FROM dldte) BETWEEN 2000 AND 2019
""")
bk["failed_fyear"] = bk["failed_fyear"].astype(int)

failed_set = set(zip(bk["gvkey"], bk["failed_fyear"]))

fin["status_label"] = fin.apply(
    lambda r: "failed" if (r["gvkey"], r["fyear"]) in failed_set else "alive", axis=1
)
print(fin["status_label"].value_counts())

status_label
alive     96954
failed        1
Name: count, dtype: int64


In [67]:
# ── Build 8-year rolling lag windows per company (mirrors the LSTM input shape) ──
FEAT_COLS = ["act","at","cogs","dltt","dp","ebit","oibdp","gp",
             "invt","lct","ni","re","rect","revt","mktval","lt","sale","xopr"]

def attach_lags(grp):
    parts = [grp]
    for lag in range(1, 8):
        shifted = grp[FEAT_COLS].shift(lag).add_prefix(f"lag{lag}_")
        parts.append(shifted)
    return pd.concat(parts, axis=1)

fin_lagged = (
    fin.groupby("gvkey", group_keys=False)
       .apply(attach_lags)
       .dropna(subset=[f"lag7_{FEAT_COLS[0]}"])   # require full 8-year window
       .reset_index(drop=True)
)
print(fin_lagged.shape)

/var/folders/mj/l0ltx7wx3md22z2ps917d6wm0000gn/T/ipykernel_87565/3289909387.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(attach_lags)


(33251, 152)


In [68]:
# ── Aggregate Glassdoor reviews → one row per (firm, fyear) ──────────────
gd = pd.read_csv("glassdoor_reviews.csv", low_memory=False, parse_dates=["date_review"])
gd["fyear"] = gd["date_review"].dt.year.astype(int)

# diversity_inclusion dropped: 83.8% null in raw data (added to platform ~2014)
# culture_values kept but imputed below: 22.8% raw null, 16.2% at cell level
RATING_COLS = ["overall_rating", "work_life_balance", "culture_values",
               "career_opp", "comp_benefits", "senior_mgmt"]

gd["_text"] = (gd["headline"].fillna("") + " " +
               gd["pros"].fillna("") + " " +
               gd["cons"].fillna("")).str.strip()

agg = (
    gd.groupby(["firm", "fyear"])
      .agg(
          **{c: (c, "mean") for c in RATING_COLS},
          review_count=("overall_rating", "count"),
          gd_text=("_text", lambda x: " ".join(x)[:100_000]),
      )
      .reset_index()
)

# Impute sparse subcategory nulls with their column means.
# overall_rating is 0% null — no imputation needed.
# The others are 0.3–16% null (Glassdoor added subcategories gradually).
for col in RATING_COLS[1:]:
    agg[col] = agg[col].fillna(agg[col].mean())

agg["fyear"] = agg["fyear"].astype(int)

assert agg[RATING_COLS].isnull().any().any() == False, "Still NaN after imputation"
print(f"agg shape: {agg.shape}  |  any NaN in rating cols: False ✓")
print(agg[["firm","fyear","review_count","overall_rating","culture_values"]].head(4))

agg shape: (4446, 10)  |  any NaN in rating cols: False ✓
                    firm  fyear  review_count  overall_rating  culture_values
0  AFH-Wealth-Management   2015             2        2.000000        2.000000
1  AFH-Wealth-Management   2016             7        2.857143        2.428571
2  AFH-Wealth-Management   2017            16        3.187500        3.066667
3  AFH-Wealth-Management   2018             6        3.666667        3.500000


In [69]:
# ── Map Glassdoor firms → Compustat gvkey (exact ticker lookup only) ──────
# No fuzzy matching — every mapping is an explicit, verified ticker.
# Anything not in TICKER_MAP is simply dropped.

TICKER_MAP = {
    "Accenture":               "ACN",
    "American-Express":        "AXP",
    "Aon":                     "AON",
    "Apple":                   "AAPL",
    "AstraZeneca":             "AZN",    # NYSE ADR
    "BAT":                     "BTI",    # NYSE ADR
    "Barclays":                "BCS",    # NYSE ADR
    "BNY-Mellon":              "BK",
    "BP":                      "BP",     # NYSE ADR
    "Booking-com":             "PCLN",   # renamed BKNG Feb 2018; PCLN covers 1999-2017
    "CBRE":                    "CBRE",
    "Cisco-Systems":           "CSCO",
    "Citi":                    "C",
    "Deutsche-Bank":           "DB",     # NYSE ADR
    "Diageo":                  "DEO",    # NYSE ADR
    "Facebook":                "FB",     # NASDAQ 2012-2018
    "GlaxoSmithKline":         "GSK",    # NYSE ADR
    "Goldman-Sachs":           "GS",
    "Google":                  "GOOGL",
    "Hilton":                  "HLT",    # NYSE 2013-2018
    "HSBC-Holdings":           "HSBC",   # NYSE ADR
    "Hyatt":                   "H",
    "IBM":                     "IBM",
    "IHG-Hotels-and-Resorts":  "IHG",    # NYSE ADR
    "Iron-Mountain-Inc":       "IRM",
    "J-P-Morgan":              "JPM",
    "JLL":                     "JLL",
    "KKR":                     "KKR",
    "Korn-Ferry":              "KFY",
    "Lloyds-Banking-Group":    "LYG",    # NYSE ADR
    "Manpower":                "MAN",
    "Marriott-International":  "MAR",
    "Mastercard":              "MA",
    "McDonald-s":              "MCD",
    "Microsoft":               "MSFT",
    "Morgan-Stanley":          "MS",
    "Oracle":                  "ORCL",
    "Pearson":                 "PSO",    # NYSE ADR
    "SAP":                     "SAP",    # NYSE ADR
    "Salesforce":              "CRM",
    "Santander":               "SAN",    # NYSE ADR
    "Sotheby-s":               "BID",    # NYSE (went private 2019)
    "Thomson-Reuters":         "TRI",
    "Unilever":                "UL",     # NYSE ADR
    "VMware":                  "VMW",
    "Vodafone":                "VOD",    # NASDAQ ADR
    "Willis-Towers-Watson":    "WLTW",   # NASDAQ
    "Wipro":                   "WIT",    # NYSE ADR
    "Workday":                 "WDAY",
    "XPO-Logistics":           "XPO",
}

# ticker → gvkey (most recent entry per ticker)
tic_to_gvkey = (
    fin_lagged[["gvkey", "tic"]]
    .drop_duplicates("tic", keep="last")
    .set_index("tic")["gvkey"]
    .to_dict()
)

# Verify every ticker resolves — list any misses so they can be fixed
no_match = {firm: tic for firm, tic in TICKER_MAP.items() if tic not in tic_to_gvkey}
if no_match:
    print("⚠ Tickers not found in Compustat pull (dropped from join):")
    for firm, tic in no_match.items():
        print(f"  {tic:8s} ← {firm}")
else:
    print("All tickers resolved in Compustat data ✓")

# Assign gvkey — anything not in TICKER_MAP gets None (dropped at join)
agg["gvkey"] = agg["firm"].map(
    {firm: tic_to_gvkey.get(tic) for firm, tic in TICKER_MAP.items()}
)

n_firms = agg["gvkey"].notna().sum()
print(f"\nMapped {agg['gvkey'].notna().any() and agg.loc[agg['gvkey'].notna(), 'firm'].nunique()} firms"
      f" → {n_firms} company-year rows available for join")

⚠ Tickers not found in Compustat pull (dropped from join):
  AXP      ← American-Express
  BCS      ← Barclays
  BK       ← BNY-Mellon
  PCLN     ← Booking-com
  C        ← Citi
  DB       ← Deutsche-Bank
  FB       ← Facebook
  GS       ← Goldman-Sachs
  HSBC     ← HSBC-Holdings
  JPM      ← J-P-Morgan
  KKR      ← KKR
  LYG      ← Lloyds-Banking-Group
  MS       ← Morgan-Stanley
  SAN      ← Santander
  WLTW     ← Willis-Towers-Watson

Mapped 35 firms → 468 company-year rows available for join


In [70]:
# ── Build output datasets ─────────────────────────────────────────────────
import os

fin_lagged["fyear"] = fin_lagged["fyear"].astype(int)

agg_matched = (
    agg.dropna(subset=["gvkey"])
    .assign(fyear=lambda d: d["fyear"].astype(int))
    .drop(columns=["firm"])
)

# master_trimodal : inner join — every row has all 3 signals, zero NaN
master_tri = (
    fin_lagged
    .merge(agg_matched.drop(columns=["gd_text"]), on=["gvkey", "fyear"], how="inner")
    .rename(columns={"conm": "company_name"})
)

# master_financial : full 33K-row universe for MLP / LSTM pretraining
master_fin = fin_lagged.rename(columns={"conm": "company_name"})

# glassdoor_text : (gvkey, fyear, gd_text) for FinBERT encoding separately
gd_text_df = agg_matched[["gvkey", "fyear", "gd_text"]].copy()

# ── Verify zero NaN in every column of the trimodal dataset ──────────────
null_counts = master_tri.isnull().sum()
bad_cols = null_counts[null_counts > 0]
if bad_cols.empty:
    print("master_trimodal: zero NaN in all columns ✓")
else:
    print("Columns with NaN (fix before training):")
    print(bad_cols)

print(f"\nmaster_trimodal  : {master_tri.shape}  — {master_tri['gvkey'].nunique()} companies")
print(f"master_financial : {master_fin.shape}")
print(f"glassdoor_text   : {gd_text_df.shape}")
print(f"\nstatus_label (trimodal):\n{master_tri['status_label'].value_counts()}")
print(f"\nRows per company:")
print(
    master_tri.groupby("company_name")["fyear"]
    .agg(n_years="count", first="min", last="max")
    .sort_values("n_years", ascending=False)
    .to_string()
)

# ── Save — delete stale files first ──────────────────────────────────────
for fname in ["master_trimodal.parquet", "master_financial.parquet",
              "glassdoor_text.parquet", "master_dataset.parquet"]:
    if os.path.exists(fname):
        os.remove(fname)

master_tri.to_parquet("master_trimodal.parquet", index=False, engine="pyarrow")
master_fin.to_parquet("master_financial.parquet", index=False, engine="pyarrow")
gd_text_df.to_parquet("glassdoor_text.parquet",  index=False, engine="pyarrow")

for fname in ["master_trimodal.parquet", "master_financial.parquet", "glassdoor_text.parquet"]:
    shape = pd.read_parquet(fname).shape
    print(f"  ✓ {fname}: {shape}")

Columns with NaN (fix before training):
dltt            4
invt            3
rect           11
lt              4
mktval          2
lag1_dltt       4
lag1_invt       2
lag1_rect      11
lag1_mktval     2
lag1_lt         4
lag2_dltt       4
lag2_invt       2
lag2_re         1
lag2_rect      11
lag2_mktval     2
lag2_lt         4
lag3_act        1
lag3_dltt       5
lag3_invt       2
lag3_lct        1
lag3_re         1
lag3_rect      10
lag3_mktval     2
lag3_lt         4
lag4_act        2
lag4_dltt       5
lag4_invt       2
lag4_lct        2
lag4_re         1
lag4_rect       9
lag4_mktval     2
lag4_lt         4
lag5_act        2
lag5_dltt       5
lag5_lct        2
lag5_re         1
lag5_rect       8
lag5_mktval     3
lag5_lt         4
lag6_act        1
lag6_dltt       5
lag6_lct        1
lag6_re         1
lag6_rect       7
lag6_mktval     9
lag6_lt         4
lag7_dltt       3
lag7_re         1
lag7_rect       6
lag7_mktval    16
lag7_lt         2
dtype: int64

master_trimodal  : (324, 159

,gvkey,cik,tic,company_name,sich,fyear,datadate,act,at,cogs,...,lag7_lt,lag7_sale,lag7_xopr,overall_rating,work_life_balance,culture_values,career_opp,comp_benefits,senior_mgmt,review_count
0,001004,0000001750,AIR,AAR CORP,5080,2006,2007-05-31,645.721,1067.633,837.171,...,401.483,1024.333,935.302,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,001004,0000001750,AIR,AAR CORP,5080,2007,2008-05-31,783.431,1362.01,1080.895,...,361.642,874.255,809.888,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,001004,0000001750,AIR,AAR CORP,5080,2008,2009-05-31,851.312,1377.511,1110.677,...,399.964,638.721,611.514,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,001004,0000001750,AIR,AAR CORP,5080,2009,2010-05-31,863.429,1501.042,1065.902,...,391.633,606.337,575.592,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,001004,0000001750,AIR,AAR CORP,5080,2010,2011-05-31,913.985,1703.727,1408.071,...,407.608,651.958,604.467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
